In [1]:
# ===== BOOTSTRAP: run first after every Colab start/restart =====
import os, shutil, subprocess
from pathlib import Path
from google.colab import drive, userdata

# 1) Reconnect Google Drive (permanent storage)
drive.mount('/content/drive')

# 2) Load secrets from the Colab Secrets panel into environment variables
SECRETS = ["GITHUB_USERNAME", "GITHUB_TOKEN", "GIT_EMAIL", "DAGSHUB_USERNAME",
           "DAGSHUB_TOKEN", "KAGGLE_API_TOKEN"]
missing = []
for k in SECRETS:
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        missing.append(k)
print(f"Secrets loaded: {len(SECRETS)-len(missing)}/{len(SECRETS)}  missing: {missing or 'none'}")

# 3) Project paths
PROJECT     = "23L-0951_A2_Ops"
DRIVE       = Path("/content/drive/MyDrive")
PROJECT_DIR = DRIVE / PROJECT              # git repo (code, notebooks, small files)
WORK_DIR    = DRIVE / f"{PROJECT}_work"    # big files kept OUT of git
LOCAL_DATA  = Path("/content/data")        # fast local copy, rebuilt each session
for p in (WORK_DIR, LOCAL_DATA):
    p.mkdir(parents=True, exist_ok=True)
os.environ.update(PROJECT_DIR=str(PROJECT_DIR), WORK_DIR=str(WORK_DIR),
                  LOCAL_DATA=str(LOCAL_DATA))

def sh(cmd, cwd=None):
    """Run a shell command and print its output. Returns the exit code."""
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)
    return r.returncode

# 4) Git settings (live on the VM, so set again every session).
#    The token is read from an env var, never saved in the repo or remote URL.
helper = '!f() { echo "username=${GITHUB_USERNAME}"; echo "password=${GITHUB_TOKEN}"; }; f'
for key, val in [("credential.helper", helper), ("user.name", "Zayan"),
                 ("user.email", os.environ.get("GIT_EMAIL", "")),
                 ("init.defaultBranch", "main")]:
    subprocess.run(["git", "config", "--global", key, val], check=True)

# 5) Clone the repo into Drive the first time; afterwards just pull updates
if not (PROJECT_DIR / ".git").exists():
    sh(f'git clone https://github.com/{os.environ["GITHUB_USERNAME"]}/{PROJECT}.git "{PROJECT_DIR}"')
os.chdir(PROJECT_DIR)
sh("rm -f .git/index.lock")      # clears a leftover lock from a dropped session
sh("git pull --ff-only")

# 6) Reinstall pinned packages (file is created in Step 2)
if (PROJECT_DIR / "requirements-colab.txt").exists():
    sh("pip install -q -r requirements-colab.txt")

# 7) Helper to save a step: copy notebook into repo, commit, push
def save_step(message, notebook_name=None):
    if notebook_name:
        src = DRIVE / "Colab Notebooks" / notebook_name
        (PROJECT_DIR / "notebooks").mkdir(exist_ok=True)
        if src.exists():
            shutil.copy(src, PROJECT_DIR / "notebooks" / notebook_name)
        else:
            print("Notebook not found:", src)
    sh("git add -A")
    sh(f'git commit -m "{message}"')
    sh("git push origin main")
    sh("git log --oneline -1")

print("Bootstrap done. Working directory:", os.getcwd())

Mounted at /content/drive
Secrets loaded: 6/6  missing: none
Cloning into '/content/drive/MyDrive/23L-0951_A2_Ops'...
Already up to date.
Bootstrap done. Working directory: /content/drive/MyDrive/23L-0951_A2_Ops


In [2]:
# Folder layout. Git ignores empty folders, so each gets an empty .gitkeep file.
for d in ["data", "src", "scripts", "notebooks", "app", "models", "tests", "reports/figures"]:
    p = PROJECT_DIR / d
    p.mkdir(parents=True, exist_ok=True)
    if not any(p.iterdir()):
        (p / ".gitkeep").touch()

# Extra .gitignore rules, appended once below GitHub's Python template
MARKER = "# --- 23L-0951_A2_Ops project rules ---"
gi = PROJECT_DIR / ".gitignore"
text = gi.read_text() if gi.exists() else ""
if MARKER not in text:
    gi.write_text(text + f"""
{MARKER}
# Secrets: never committed
.env
kaggle.json
access_token
# Local MLflow store (real tracking is on DagsHub)
mlruns/
mlartifacts/
# Checkpoints / notebook / OS junk
checkpoints/
.ipynb_checkpoints/
.DS_Store
""")

# .env.example lists WHICH variables are needed, with fake values.
(PROJECT_DIR / ".env.example").write_text("""# Copy to .env and fill in real values. .env is git-ignored.
DAGSHUB_USERNAME=your_dagshub_username
DAGSHUB_TOKEN=your_dagshub_token
MLFLOW_TRACKING_URI=https://dagshub.com/your_dagshub_username/23L-0951_A2_Ops.mlflow
MLFLOW_TRACKING_USERNAME=your_dagshub_username
MLFLOW_TRACKING_PASSWORD=your_dagshub_token
""")

(PROJECT_DIR / "README.md").write_text("""# VisionGuard (23L-0951_A2_Ops)

Classifies an uploaded image as **AI-generated** or **real**.
MLOps Assignment 2, FAST NUCES Lahore.

| Part | Tool |
|---|---|
| Data versioning | DVC + DagsHub storage |
| Experiments & registry | MLflow on DagsHub |
| Web app | Flask (`app/`) |
| Containers | Docker Compose |

_Sections are filled in as the project progresses._
""")

for p in sorted(PROJECT_DIR.rglob("*")):
    if ".git" not in p.parts:
        print(p.relative_to(PROJECT_DIR))

.env.example
.gitignore
README.md
app
app/.gitkeep
data
data/.gitkeep
models
models/.gitkeep
notebooks
notebooks/.gitkeep
reports
reports/figures
reports/figures/.gitkeep
scripts
scripts/.gitkeep
src
src/.gitkeep
tests
tests/.gitkeep
